<a href='https://www.darshan.ac.in/'> <img src='https://www.darshan.ac.in/Content/media/DU_Logo.svg' width="250" height="300"/></a>
<pre>
<center><b><h1>Machine Learning - 2301CS621</b></center>
    <center><b><h1>Jenisa Vasani - 24010101903</b></center>

<center><b><h1>Lab - 2</b></center>    
<pre>    

# EDA & Pipeline: Google Play Store Apps

**Dataset:** Google Play Store Apps (Available on Kaggle) <BR>
**Objective:** Transform raw, messy data into clean, actionable insights using Pandas and Scikit-Learn pipelines.<BR>
**Focus:** Data Cleaning, String Sanitization, Advanced Imputation, Correlation, and Pipelines.<BR>

### 1. Setup & Initialization

**Exercise 1: Import Dependencies**
* Import `pandas`, `numpy`, `matplotlib.pyplot`, and `seaborn`.
* Set pandas options to display all columns (visual aid).

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns



**Exercise 2: Data Loading & Initial Inspection**
* Load the `googleplaystore.csv` file.
* Display the first 5 rows.
* **Check:** Look closely at the `Installs`, `Size`, and `Price` columns. Notice they are currently Objects (strings), not numbers.

In [ ]:
df = pd.read_csv('googleplaystore.csv')
df

In [ ]:
df.dtypes


### 2. Data Integrity Check

**Exercise 3: Audit Data Types and Missing Values**
* Use a single command to view data types (`dtypes`) and non-null counts.
* Calculate the *percentage* of missing values for each column.

In [ ]:
df.info()


In [ ]:
percentage = (df.isnull().sum() / df.shape[0])*100
percentage

**Exercise 4: Handling Duplicates**
* Duplicate entries skew results. Check for duplicate rows.
* Drop duplicates, keeping the *first* occurrence. Verify the shape change.

In [ ]:
# For Checking Duplicates use duplicated()
# Use drop_duplicates to Drop()

print('duplicate before:',df.duplicated().sum())


In [ ]:
df.drop_duplicates(inplace=True)
print('duplicate after:',df.duplicated().sum())

### 3. Advanced String Sanitization (Crucial Step)

**Exercise 5: Cleaning the 'Installs' Column**
* The `Installs` column contains characters like `+` and `,` (e.g., "10,000+").
* Remove these characters.
* Convert the column to a numeric integer type.

In [ ]:
# use Column.astype(str).str.replace
df['Installs'] = df['Installs'].str.replace('+','')
df['Installs'] = df['Installs'].str.replace(',','')
df['Installs']

In [ ]:
# use to_numeric to convert to int
df['Installs']=pd.to_numeric(df['Installs'],errors = 'coerce') #downcast convert smallest possible numeric type
df['Installs']


In [ ]:
df['Installs'].dtypes

In [ ]:
df['Installs'].iloc[9990]

**Exercise 6: Cleaning the 'Price' Column**
* The `Price` column contains the `$` symbol (e.g., "$4.99").
* Remove the symbol.
* Convert the column to a `float`.

In [ ]:
df['Price'] = df['Price'].str.replace('$',"")
df['Price']

In [ ]:
df['Price'].dtype
df['Price']=pd.to_numeric(df['Price'],errors='coerce')##error = ignore : Invalid parsing will return the input unchanged.
df['Price']

In [ ]:
df['Price'].iloc[9990]

In [ ]:
# Same as Above
df['Price'].dtypes

In [ ]:
df['Reviews']=pd.to_numeric(df['Reviews'],errors = 'coerce')
df['Reviews']

**Exercise 7: Complex Logic - Sanitizing 'Size'**
* The `Size` column is messy. It contains 'M' (Megabytes), 'k' (kilobytes), and string 'Varies with device'.
* **Task:** Write a function (or apply lambda) to:
    1.  Replace 'k' with 'e+3' and 'M' with 'e+6'.
    2.  Coerce 'Varies with device' to `NaN`.
    3.  Convert the string to a number.

In [ ]:
# Hint: Define a function clean_size(x).
# Hint: If 'M' in x: return float(x.replace('M', '')) * 1000000
# Hint: Handle the 'Varies with device' edge case carefully.

def clean_size(x):
    x = str(x)
    if 'M' in x:
        # Convert 19M to 19000000
        return float(x.replace('M', '')) * 1000000
        
    elif 'k' in x:
        # Convert 500k to 500000
        return float(x.replace('k', '')) * 1000
        
    elif 'Varies with device' in x:
        # Handle string edge case
        return np.nan
        
    else:
        # Attempt to convert or return NaN
        try:
            return float(x)
        except:
            return np.nan

In [ ]:
# Use apply Method to apply above fun
df['Size'] = df['Size'].apply(clean_size)
df['Size']

### 4. Advanced Imputation

**Exercise 8: Analyzing Missing 'Rating'**
* The `Rating` column has missing values.
* **Visualize** the distribution of Ratings using a Histogram or KDE plot to decide between Mean vs Median imputation.

In [ ]:
df['Rating'].isnull().sum()

In [ ]:
mean_value = df['Rating'].mean()
df_mean = df.copy()
df_mean['Rating'] = df_mean['Rating'].fillna(mean_value)
df_mean['Rating']

In [ ]:
median_value = df['Rating'].median()
df_median = df.copy()
df_median['Rating'] = df_median['Rating'].fillna(median_value)
df_median['Rating']

In [ ]:
plt.figure(figsize=(14,6))


plt.subplot(1,2,1)
sns.histplot(df_mean['Rating'], kde=True)
plt.title("Ratings After Mean Imputation")


plt.subplot(1,2,2)
sns.histplot(df_median['Rating'], kde=True)
plt.title("Ratings After Median Imputation")

plt.show()

**Exercise 9: Group-Specific Imputation**
* Fill missing `Rating` values with the **Median Rating** of the specific `Category` the app belongs to.
* *Example:* If a "Business" app is missing a rating, fill it with the median rating of all "Business" apps.

In [ ]:
df['Rating'] = df.groupby('Category')['Rating']

#After this use transform Method

**Exercise 10: Drop Remaining NaNs**
* For the remaining columns with minimal missing data (like `Current Ver`), simply drop the rows containing NaNs to ensure a clean dataset for correlation.

In [ ]:
# dropna

### 5. Correlation & Visualization

**Exercise 11: Correlation Matrix**
* Generate a correlation matrix for the numerical columns (`Rating`, `Reviews`, `Size`, `Installs`, `Price`).

**Exercise 12: Heatmap Visualization**
* Visualize the correlation matrix using a Seaborn Heatmap.
* Annotate the values.

**Exercise 13: Scatter Plot Analysis**
* Create a Scatter Plot to analyze the relationship between `Reviews` and `Installs`.
* **Note:** You might need to use a log scale for the axes if the data is skewed.

In [ ]:
# If needed
plt.xscale('log')
plt.yscale('log')
plt.title('Reviews vs Installs (Log Scale)')
plt.show()

**Exercise 14: Categorical Aggregation**
* Create a Bar Plot showing the top 10 Categories by **Total Installs**.

In [ ]:

top_cats = df.groupby('Category')['Installs']
